# P6 XER Analysis — Cosedil PMO
**Notebook di analisi Primavera P6 via file XER**  
Parsare XER da file `.xer` diretto o da Excel (drag & drop), con recipe book completo per scheduling, WBS, legami, critical path, UDF, Activity Codes e bridge CPM→P6.

> Generato dalla skill `p6-xer-analysis` — Luca Savatta, Cosedil S.p.A.

## 0. Setup
Installa le dipendenze se necessario e importa le librerie.

In [ ]:
# !pip install pandas openpyxl xlrd  # decommentare se necessario

import pandas as pd
import re
from io import StringIO
from pathlib import Path

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Parser XER
### 1a. Da file `.xer` diretto

In [ ]:
def parse_xer(filepath: str) -> dict:
    """Legge un file .xer e restituisce un dict di DataFrame per tabella."""
    with open(filepath, encoding='latin-1') as f:
        content = f.read()
    return parse_xer_from_string(content)


def parse_xer_from_string(content: str) -> dict:
    tables = {}
    blocks = re.split(r'(?=^%T\t)', content, flags=re.MULTILINE)
    for block in blocks:
        lines = [l for l in block.strip().split('\n') if l.strip()]
        if not lines or not lines[0].startswith('%T\t'):
            continue
        table_name = lines[0].split('\t')[1].strip()
        fields, rows = None, []
        for line in lines[1:]:
            if line.startswith('%F\t'):
                fields = line[3:].split('\t')
            elif line.startswith('%R\t'):
                rows.append(line[3:].split('\t'))
        if fields and rows:
            rows = [r + [''] * (len(fields) - len(r)) for r in rows]
            tables[table_name] = pd.DataFrame(
                [r[:len(fields)] for r in rows], columns=fields
            )
    return tables


# Uso:
# xer = parse_xer('percorso/progetto.xer')
# print(list(xer.keys()))  # tabelle disponibili

### 1b. Da Excel (XER drag & drop su foglio)
Quando il `.xer` è stato trascinato direttamente su Excel, il foglio mantiene la struttura raw `%T/%F/%R`.

In [ ]:
def parse_xer_from_excel(xlsx_path: str, sheet_name=0) -> dict:
    """Parsare XER importato su Excel come foglio raw."""
    raw = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, dtype=str)
    content = '\n'.join(
        '\t'.join('' if pd.isna(c) else str(c).strip() for c in row)
        for _, row in raw.iterrows()
    )
    return parse_xer_from_string(content)


# Uso:
# xer = parse_xer_from_excel('3123_CEMEX_2026.xlsx')
# print({k: v.shape for k, v in xer.items()})

## 2. Ispezione rapida delle tabelle

In [ ]:
# Carica il tuo file — modifica il percorso:
XER_PATH = 'progetto.xer'           # oppure un .xlsx con XER raw

# xer = parse_xer(XER_PATH)
# xer = parse_xer_from_excel(XER_PATH)

# Panoramica tabelle disponibili e dimensioni
# for name, df in xer.items():
#     print(f'{name:20s}  {df.shape[0]:>6} righe  {df.shape[1]:>3} colonne')

## 3. Flat view — TASK + WBS
Unisce la tabella `TASK` con `PROJWBS` e calcola durate/float in giorni.

In [ ]:
def build_task_view(xer: dict) -> pd.DataFrame:
    task = xer['TASK'].copy()
    wbs  = xer['PROJWBS'][['wbs_id', 'wbs_short_name', 'wbs_name']].rename(
        columns={'wbs_short_name': 'wbs_code', 'wbs_name': 'wbs_descr'})

    df = task.merge(wbs, on='wbs_id', how='left')

    # Conversioni numeriche
    for col in ['total_float_hr_cnt', 'free_float_hr_cnt', 'remain_drtn_hr_cnt',
                'target_drtn_hr_cnt', 'phys_complete_pct']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['total_float_gg']    = df['total_float_hr_cnt']   / 8
    df['durata_target_gg']  = df['target_drtn_hr_cnt']   / 8
    df['durata_residua_gg'] = df['remain_drtn_hr_cnt']   / 8
    df['is_critical']       = df['total_float_hr_cnt']  == 0

    # Conversione date
    date_cols = ['early_start_date', 'early_end_date', 'late_start_date', 'late_end_date',
                 'target_start_date', 'target_end_date', 'act_start_date', 'act_end_date']
    for c in date_cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')

    return df


# task_view = build_task_view(xer)
# task_view[['task_code','task_name','wbs_code','status_code',
#            'durata_target_gg','total_float_gg','is_critical',
#            'early_start_date','early_end_date']].head(20)

## 4. Critical Path

In [ ]:
def critical_path_summary(task_view: pd.DataFrame) -> pd.DataFrame:
    """Filtra solo attività critiche di tipo normale (no WBS summary, no milestone)."""
    crit = task_view[
        task_view['is_critical'] & (task_view['task_type'] == 'TT_Task')
    ].copy()
    cols = ['task_code', 'task_name', 'wbs_code', 'wbs_descr',
            'early_start_date', 'early_end_date',
            'durata_target_gg', 'durata_residua_gg',
            'phys_complete_pct', 'status_code']
    return crit[cols].sort_values('early_start_date').reset_index(drop=True)


# crit = critical_path_summary(task_view)
# print(f'Attività critiche: {len(crit)}')
# crit.head(20)

## 5. Analisi Legami (TASKPRED)

In [ ]:
def build_logic_view(xer: dict) -> pd.DataFrame:
    """Restituisce i legami con nomi attività successore/predecessore."""
    pred     = xer['TASKPRED'].copy()
    task_ref = xer['TASK'][['task_id', 'task_code', 'task_name']].copy()

    pred['lag_hr_cnt'] = pd.to_numeric(pred['lag_hr_cnt'], errors='coerce')
    pred['lag_gg']     = pred['lag_hr_cnt'] / 8

    # Mappa tipo legame P6 → sigla leggibile
    pred_type_map = {
        'PR_FS': 'FS', 'PR_SS': 'SS', 'PR_FF': 'FF', 'PR_SF': 'SF'
    }
    pred['tipo_legame'] = pred['pred_type'].map(pred_type_map).fillna(pred['pred_type'])

    df = pred.merge(
        task_ref.rename(columns={'task_code': 'succ_code', 'task_name': 'succ_name'}),
        on='task_id', how='left'
    )
    df = df.merge(
        task_ref.rename(columns={'task_id': 'pred_task_id',
                                  'task_code': 'pred_code', 'task_name': 'pred_name'}),
        on='pred_task_id', how='left'
    )
    return df[['succ_code', 'succ_name', 'tipo_legame', 'lag_gg', 'pred_code', 'pred_name']]


# logic = build_logic_view(xer)
# # Filtra legami con lag > 0
# logic[logic['lag_gg'] > 0].sort_values('lag_gg', ascending=False).head(20)

## 6. Slippage — Baseline vs Current
Confronta due XER (baseline e current) sullo stesso set di `task_code`.

In [ ]:
def slippage_analysis(task_current: pd.DataFrame,
                       task_baseline: pd.DataFrame) -> pd.DataFrame:
    """Calcola lo scostamento in giorni tra early_end_date corrente e target baseline."""
    cur = task_current[['task_code', 'task_name', 'early_end_date',
                         'phys_complete_pct', 'status_code']].copy()
    bas = task_baseline[['task_code', 'target_end_date']].copy()

    df = cur.merge(bas, on='task_code', how='inner')
    df['slippage_gg'] = (df['early_end_date'] - df['target_end_date']).dt.days
    df['stato_scost'] = df['slippage_gg'].apply(
        lambda x: 'ANTICIPO' if x < -1 else ('IN LINEA' if abs(x) <= 1 else 'RITARDO')
    )
    return df.sort_values('slippage_gg', ascending=False).reset_index(drop=True)


# xer_baseline = parse_xer('baseline.xer')
# xer_current  = parse_xer('current.xer')
# tv_bas = build_task_view(xer_baseline)
# tv_cur = build_task_view(xer_current)
# slip = slippage_analysis(tv_cur, tv_bas)
# slip[slip['stato_scost'] == 'RITARDO'].head(20)

## 7. User Defined Fields (UDF)
Campi custom P6 — nel progetto CEMEX: `CEMEX TAG COMPONENTE`, `CEMEX N COMPONENTE`, `quantity`, `CEMEX 2025 - Activity code Sogin`.

In [ ]:
def get_udf_values(xer: dict) -> pd.DataFrame:
    """Pivot UDF: una riga per task_id, una colonna per label UDF."""
    udf_types = xer['UDFTYPE'][['udf_type_id', 'udf_type_label']].copy()
    udf_vals  = xer['UDFVALUE'].copy()

    df = udf_vals.merge(udf_types, on='udf_type_id', how='left')

    # Valore: preferisce udf_text, fallback udf_float
    if 'udf_text' in df.columns:
        df['value'] = df['udf_text'].where(df['udf_text'].notna() & (df['udf_text'] != ''),
                                            df.get('udf_float', ''))
    else:
        df['value'] = df.get('udf_float', '')

    pivot = df.pivot_table(
        index='fk_id', columns='udf_type_label',
        values='value', aggfunc='first'
    ).reset_index().rename(columns={'fk_id': 'task_id'})

    return pivot


# udf = get_udf_values(xer)
# udf.head()

## 8. Activity Codes
Mapping `task_id` → codici attività (Lotto, Disciplina, Opera, etc.).

In [ ]:
def get_activity_codes(xer: dict) -> pd.DataFrame:
    """Restituisce un pivot task_id → activity codes (una colonna per tipo)."""
    actv_types = xer.get('ACTVTYPE', pd.DataFrame())
    actv_codes = xer.get('ACTVCODE', pd.DataFrame())
    task_actv  = xer.get('TASKACTV', pd.DataFrame())

    if actv_codes.empty or task_actv.empty:
        print('Nessun activity code trovato nel file XER.')
        return pd.DataFrame()

    codes_with_type = actv_codes.merge(
        actv_types[['actv_code_type_id', 'actv_code_type']],
        on='actv_code_type_id', how='left'
    )
    df = task_actv.merge(
        codes_with_type[['actv_code_id', 'actv_code_type', 'short_name']],
        on='actv_code_id', how='left'
    )
    pivot = df.pivot_table(
        index='task_id', columns='actv_code_type',
        values='short_name', aggfunc='first'
    ).reset_index()
    return pivot


# actv = get_activity_codes(xer)
# actv.head()

## 9. Vista completa arricchita
Unisce TASK, WBS, UDF e Activity Codes in un unico DataFrame da esportare.

In [ ]:
def build_full_view(xer: dict) -> pd.DataFrame:
    """Task view completa: WBS + UDF + Activity Codes."""
    df = build_task_view(xer)

    # Activity codes
    actv = get_activity_codes(xer)
    if not actv.empty:
        df = df.merge(actv, on='task_id', how='left')

    # UDF
    if 'UDFVALUE' in xer and 'UDFTYPE' in xer:
        udf = get_udf_values(xer)
        df = df.merge(udf, on='task_id', how='left')

    return df


# full = build_full_view(xer)
# print(f'Shape: {full.shape}')
# full.columns.tolist()

## 10. Bridge CPM → P6
Distribuzione importi SIL su attività P6: `Articolo → Cod.WBS → Activity ID → peso proporzionale`.

In [ ]:
def build_bridge(sil_df: pd.DataFrame,
                  budget_df: pd.DataFrame,
                  p6_task_df: pd.DataFrame,
                  mapping_df: pd.DataFrame) -> pd.DataFrame:
    """
    Logica bridge CPM TeamSystem → Primavera P6.

    Parameters
    ----------
    sil_df      : foglio 'SIL diretti' (colonne: Articolo, Importo, Cod. WBS, ...)
    budget_df   : foglio 'BUDGET'      (colonne: Articolo, Cod. WBS)
    p6_task_df  : task view P6         (colonne: task_code, wbs_id, act_cost, target_drtn_hr_cnt)
    mapping_df  : foglio 'MAPPING'     (colonne: Cod. WBS, Activity ID P6)
    """
    # Step 1: SIL Articolo → WBS CPM
    sil_wbs = sil_df.merge(
        budget_df[['Articolo', 'Cod. WBS']].drop_duplicates(),
        on='Articolo', how='left'
    )

    # Step 2: WBS CPM → Activity ID P6
    sil_mapped = sil_wbs.merge(
        mapping_df[['Cod. WBS', 'Activity ID P6']],
        on='Cod. WBS', how='left'
    )

    # Step 3: Peso proporzionale ad act_cost; fallback su durata se act_cost = 0
    p6 = p6_task_df.copy()
    p6['act_cost']           = pd.to_numeric(p6['act_cost'], errors='coerce').fillna(0)
    p6['target_drtn_hr_cnt'] = pd.to_numeric(p6['target_drtn_hr_cnt'], errors='coerce').fillna(0)

    wbs_cost_sum = p6.groupby('wbs_id')['act_cost'].transform('sum')
    wbs_drtn_sum = p6.groupby('wbs_id')['target_drtn_hr_cnt'].transform('sum')

    p6['weight'] = p6['act_cost'] / wbs_cost_sum.where(wbs_cost_sum > 0, 1)
    # Fallback durata dove act_cost = 0
    mask_no_cost = wbs_cost_sum == 0
    p6.loc[mask_no_cost, 'weight'] = (
        p6.loc[mask_no_cost, 'target_drtn_hr_cnt'] /
        wbs_drtn_sum[mask_no_cost].where(wbs_drtn_sum[mask_no_cost] > 0, 1)
    )

    result = sil_mapped.merge(
        p6[['task_code', 'weight', 'wbs_id']],
        on='wbs_id', how='left'
    )
    result['importo_distribuito'] = result['Importo'] * result['weight'].fillna(0)
    return result


# Uso con il file CPM_to_P6:
# cpm   = pd.read_excel('CPM_to_P6_rev_0_1.xlsx', sheet_name=None)
# sil   = cpm['SIL diretti']
# bud   = cpm['BUDGET']
# mapp  = cpm['MAPPING']
# xer   = parse_xer_from_excel('CPM_to_P6_rev_0_1.xlsx', sheet_name='INPUT_P6')
# tv    = build_task_view(xer)
# bridge = build_bridge(sil, bud, tv, mapp)
# bridge.head()

## 11. Export risultati su Excel
Esporta le viste principali in un unico file `.xlsx` con fogli separati.

In [ ]:
def export_analysis(xer: dict, output_path: str = 'P6_Analysis_Output.xlsx'):
    """Esporta task view, critical path e legami su Excel multisheet."""
    task_view = build_task_view(xer)
    crit      = critical_path_summary(task_view)
    logic     = build_logic_view(xer)

    cols_task = ['task_code', 'task_name', 'wbs_code', 'wbs_descr', 'status_code',
                 'task_type', 'phys_complete_pct', 'durata_target_gg',
                 'durata_residua_gg', 'total_float_gg', 'is_critical',
                 'early_start_date', 'early_end_date',
                 'late_start_date', 'late_end_date',
                 'target_start_date', 'target_end_date']
    cols_task = [c for c in cols_task if c in task_view.columns]

    with pd.ExcelWriter(output_path, engine='openpyxl', datetime_format='DD/MM/YYYY') as w:
        task_view[cols_task].to_excel(w, sheet_name='TASK_VIEW',   index=False)
        crit.to_excel(             w, sheet_name='CRITICAL_PATH', index=False)
        logic.to_excel(            w, sheet_name='LEGAMI',        index=False)

        # UDF se disponibili
        if 'UDFVALUE' in xer and 'UDFTYPE' in xer:
            udf = get_udf_values(xer)
            udf.to_excel(w, sheet_name='UDF_VALUES', index=False)

    print(f'Export completato → {output_path}')


# export_analysis(xer, 'P6_CEMEX_Analysis.xlsx')

## 12. Quick Stats — panoramica progetto

In [ ]:
def quick_stats(xer: dict):
    """Stampa statistiche rapide sul progetto."""
    task = xer['TASK'].copy()
    task['total_float_hr_cnt'] = pd.to_numeric(task['total_float_hr_cnt'], errors='coerce')
    task['phys_complete_pct']  = pd.to_numeric(task['phys_complete_pct'],  errors='coerce')

    only_tasks = task[task['task_type'] == 'TT_Task']

    print('=' * 50)
    print(f'  Totale attività (TT_Task):  {len(only_tasks):>6}')
    print(f'  Non iniziate  (TK_NotStart):{only_tasks[only_tasks["status_code"]=="TK_NotStart"].shape[0]:>6}')
    print(f'  In corso      (TK_Active):  {only_tasks[only_tasks["status_code"]=="TK_Active"].shape[0]:>6}')
    print(f'  Completate    (TK_Complete):{only_tasks[only_tasks["status_code"]=="TK_Complete"].shape[0]:>6}')
    print(f'  Critiche      (float=0):    {(only_tasks["total_float_hr_cnt"]==0).sum():>6}')
    print(f'  % fisica media:             {only_tasks["phys_complete_pct"].mean():>6.1f}%')
    print('=' * 50)

    # WBS con più attività critiche
    tv = build_task_view(xer)
    crit_by_wbs = (
        tv[tv['is_critical'] & (tv['task_type']=='TT_Task')]
        .groupby(['wbs_code','wbs_descr'])
        .size().reset_index(name='n_critiche')
        .sort_values('n_critiche', ascending=False)
    )
    print('\nTop 10 WBS per attività critiche:')
    print(crit_by_wbs.head(10).to_string(index=False))


# quick_stats(xer)